# Experiment no 04

## 1. Aim
To implement polynomial regression using a suitable open-access dataset, evaluate models with different polynomial degrees, and study the effect of model complexity on prediction accuracy and the bias–variance trade-off.

## 2. Objectives
1. Understand polynomial regression and its applications.
2. Use an open-access real-world dataset for regression analysis.
3. Train polynomial regression models with degrees 1, 2, 3, 5, 10, and 15.
4. Evaluate models using MSE, RMSE, and R2.
5. Compare training and testing errors to identify underfitting and overfitting.
6. Understand how polynomial degree affects complexity, bias, variance, and generalization.

## 3. Theory
Polynomial regression models a nonlinear relationship between an input variable and a continuous output by including powers of the input variable:
$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d$$
Here, $x$ is the independent variable, $\hat{y}$ is the predicted output, $\beta_i$ are model coefficients, and $d$ is the polynomial degree. Degree 1 is ordinary linear regression; higher degrees allow more flexible curves.

### Bias–variance trade-off
- **High bias / underfitting:** A model that is too simple cannot capture the underlying pattern and may have high training and testing errors.
- **Balanced model:** A suitable degree captures the main pattern while avoiding unnecessary complexity.
- **High variance / overfitting:** A very flexible model may fit training observations closely but perform poorly on unseen data.

## 4. Dataset
Dataset: **Auto MPG**, from the UCI Machine Learning Repository.
- **Input feature (`weight`):** Vehicle weight.
- **Target (`mpg`):** Fuel efficiency in miles per gallon.
- **Candidate polynomial degrees:** 1, 2, 3, 5, 10, and 15.
- **Train–test split:** 80% training and 20% testing.

## 5. Algorithm
1. Load the Auto MPG dataset.
2. Select `weight` as the input and `mpg` as the target.
3. Remove rows with missing values in these columns.
4. Split the data into training and testing subsets.
5. Standardize the input feature.
6. Generate polynomial features for each selected degree.
7. Train a linear regression model on the transformed features.
8. Predict target values for training and testing data.
9. Calculate MSE, RMSE, and R2 for each degree.
10. Plot regression curves and compare training and testing errors.
11. Select a suitable degree based on test performance and model complexity.


### Cell 1: Import libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
print("Libraries imported successfully.")


### Cell 2: Load the open-access dataset


In [ ]:
import os
import ssl
import urllib.request

local_file = "auto-mpg.data"
columns = [
    "mpg", "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year", "origin", "car_name"
]

if not os.path.exists(local_file):
    url = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/auto-mpg/auto-mpg.data"
    )
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.urlopen(url, context=ctx)
    with open(local_file, "wb") as f:
        f.write(req.read())

df = pd.read_csv(
    local_file,
    sep=r"\s+",
    names=columns,
    na_values="?",
    comment="\t",
    skipinitialspace=True
)

# Select the feature and target; drop rows missing either value.
data = df[["weight", "mpg"]].dropna().copy()

print("Dataset shape:", data.shape)
display(data.head())
print("\nMissing values:\n", data.isnull().sum())


### Cell 3: Visualize the dataset


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data["weight"], data["mpg"], alpha=0.7)
plt.xlabel("Vehicle Weight")
plt.ylabel("Fuel Efficiency (MPG)")
plt.title("Vehicle Weight vs Fuel Efficiency")
plt.grid(True, alpha=0.3)
plt.show()


### Cell 4: Split the dataset


In [ ]:
X = data[["weight"]]
y = data["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


### Cell 5: Train polynomial regression models


In [ ]:
degrees = [1, 2, 3, 5, 10, 15]

results = []
trained_models = {}

for degree in degrees:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("polynomial", PolynomialFeatures(
            degree=degree,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])
    
    model.fit(X_train, y_train)
    
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    
    results.append({
        "Degree": degree,
        "Train MSE": train_mse,
        "Test MSE": test_mse,
        "Train RMSE": np.sqrt(train_mse),
        "Test RMSE": np.sqrt(test_mse),
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred)
    })
    
    trained_models[degree] = model

results_df = pd.DataFrame(results)
display(results_df.round(4))


### Cell 6: Plot polynomial regression curves


In [ ]:
import warnings
warnings.filterwarnings("ignore")

X_plot = np.linspace(
    X["weight"].min(),
    X["weight"].max(),
    400
).reshape(-1, 1)

plt.figure(figsize=(12, 7))
plt.scatter(
    X_train["weight"], y_train,
    alpha=0.45, label="Training data"
)

for degree in degrees:
    y_plot = trained_models[degree].predict(X_plot)
    plt.plot(X_plot.ravel(), y_plot, label=f"Degree {degree}")

plt.xlabel("Vehicle Weight")
plt.ylabel("Fuel Efficiency (MPG)")
plt.title("Polynomial Regression with Different Degrees")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Cell 7: Compare training and testing RMSE


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    results_df["Degree"], results_df["Train RMSE"],
    marker="o", label="Training RMSE"
)
plt.plot(
    results_df["Degree"], results_df["Test RMSE"],
    marker="s", label="Testing RMSE"
)
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE")
plt.title("Effect of Polynomial Degree on Prediction Error")
plt.xticks(degrees)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Cell 8: Compare R2 scores


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    results_df["Degree"], results_df["Train R2"],
    marker="o", label="Training R2"
)
plt.plot(
    results_df["Degree"], results_df["Test R2"],
    marker="s", label="Testing R2"
)
plt.xlabel("Polynomial Degree")
plt.ylabel("R2 Score")
plt.title("Effect of Polynomial Degree on R2")
plt.xticks(degrees)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Cell 9: Identify the best degree by test RMSE


In [ ]:
best_row = results_df.loc[results_df["Test RMSE"].idxmin()]

print("Best degree on this test split:", int(best_row["Degree"]))
print(f"Test MSE:  {best_row['Test MSE']:.4f}")
print(f"Test RMSE: {best_row['Test RMSE']:.4f}")
print(f"Test R2:   {best_row['Test R2']:.4f}")


## 8. Evaluation Metrics
- **Mean Squared Error (MSE):**
  $$MSE = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$
- **Root Mean Squared Error (RMSE):**
  $$RMSE = \sqrt{MSE}$$
- **Coefficient of Determination ($R^2$):**
  $$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

## 9. Expected Observations
- **Degree 1 (Straight line):** May underfit a curved relationship (high bias).
- **Degree 2 (Simple curve):** Captures basic non-linearity well.
- **Degree 3-5 (More flexible):** Balances fit and complexity.
- **Degree 10-15 (Highly flexible):** Risk of high variance and wild oscillations at boundaries.

## 10. Result and Conclusion
Polynomial regression demonstrates how model complexity affects predictive performance. A low-degree polynomial may underfit the data, while a very high-degree polynomial may fit training observations too closely and generalize poorly. The most suitable degree balances flexibility and generalization.
